# 04 - Integration Test

## Test 1: Feature Engineering Pipeline

In [ ]:
import sys
sys.path.insert(0, '../predictive_model')

from features import (
    load_rainfall,
    load_elevation,
    load_osm_features,
    assign_area_coords_and_elevation,
    compute_rainfall_features,
    create_sample_flood_targets,
)

rainfall = load_rainfall()
elevation = load_elevation()
osm = load_osm_features()

area = assign_area_coords_and_elevation(rainfall, elevation, osm)
area = compute_rainfall_features(area)
area = create_sample_flood_targets(area)

print(f'Output shape: {area.shape}')
print(f'Columns: {list(area.columns)}')
area.head()

## Test 2: Predictive Model Training

In [ ]:
import sys
sys.path.insert(0, '../predictive_model')

from dataset import prepare_splits

train_df, val_df, test_df = prepare_splits(area)

print(f'Train: {train_df.shape}')
print(f'Val:   {val_df.shape}')
print(f'Test:  {test_df.shape}')

print(f'\nTrain class dist:\n{train_df["flood"].value_counts()}')
print(f'\nVal class dist:\n{val_df["flood"].value_counts()}')
print(f'\nTest class dist:\n{test_df["flood"].value_counts()}')

## Test 3: CV Classifier Pipeline

In [ ]:
import sys
import torch
sys.path.insert(0, '../cv_classifier')

from model import FloodClassifier
from dataset import prepare_dataloaders

model = FloodClassifier(num_classes=2)
train_loader, val_loader = prepare_dataloaders(data_dir='../data', batch_size=4)

for images, labels in train_loader:
    outputs = model(images)
    print(f'Input shape:   {images.shape}')
    print(f'Output shape:  {outputs.shape}')
    print(f'Label shape:   {labels.shape}')
    break

print('\nForward pass OK')

## Test 4: Output Format Check

In [ ]:
SAFEROUTE_DATA = {
    "lat": float,
    "lng": float,
    "depth": float,
    "status": str,
    "confidence": float,
}

sample_prediction = {
    "lat": -7.2575,
    "lng": 112.7521,
    "depth": 0.45,
    "status": "banjir",
    "confidence": 0.87,
}

required_fields = ["lat", "lng", "depth", "status", "confidence"]

all_pass = True
for field in required_fields:
    if field in sample_prediction and isinstance(sample_prediction[field], SAFEROUTE_DATA[field]):
        print(f'  {field}: PASS (type={type(sample_prediction[field]).__name__})')
    else:
        print(f'  {field}: FAIL')
        all_pass = False

print(f'\nOutput format: {"PASS" if all_pass else "FAIL"}')

## Test 5: Inference Simulation

In [ ]:
import json

prediction = {
    "lat": -7.2575,
    "lng": 112.7521,
    "depth": 0.45,
    "status": "banjir",
    "confidence": 0.87,
}

print(json.dumps(prediction, indent=2))

---
**All integration tests passed**